In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
import pymc as pm
import arviz as az

In [3]:
brent_df = pd.read_csv('/content/drive/MyDrive/processed/processed_brent_data.csv')
brent_df['Date'] = pd.to_datetime(brent_df['Date'])

In [6]:
plt.plot(brent_df['Date'], brent_df['Price'], label='Brent Oil Price')
plt.title('Brent Oil Price Over Time')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.savefig('/content/drive/MyDrive/plots/brent_price_plot.png')
plt.close()

In [7]:
result = adfuller(brent_df['Price'])
print('ADF Statistic:', result[0])
print('p-value:', result[1])
print('Critical Values:', result[4])

ADF Statistic: -1.9938560113924666
p-value: 0.2892735048934033
Critical Values: {'1%': np.float64(-3.4310783342658615), '5%': np.float64(-2.861861876398633), '10%': np.float64(-2.566941329781918)}


In [8]:
brent_df['Log_Returns'] = np.log(brent_df['Price']).diff()
brent_df = brent_df.dropna()

In [9]:
plt.figure(figsize=(12, 6))
plt.plot(brent_df['Date'], brent_df['Log_Returns'], label='Log Returns')
plt.title('Log Returns of Brent Oil Prices')
plt.xlabel('Date')
plt.ylabel('Log Returns')
plt.legend()
plt.savefig('/content/drive/MyDrive/plots/log_returns_plot.png')
plt.close()

In [11]:
log_returns = brent_df['Log_Returns'].values
dates = brent_df['Date'].values
n = len(log_returns)

In [12]:
with pm.Model() as model:
    tau = pm.DiscreteUniform("tau", lower=0, upper=n-1)
    mu_1 = pm.Normal("mu_1", mu=0, sigma=1)
    mu_2 = pm.Normal("mu_2", mu=0, sigma=1)

    sigma = pm.HalfNormal("sigma", sigma=1)
    mu = pm.math.switch(tau >= np.arange(n), mu_1, mu_2)

    likelihood = pm.Normal("likelihood", mu=mu, sigma=sigma, observed=log_returns)

    trace = pm.sample(2000, tune=1000, return_inferencedata=True)

summary = az.summary(trace)
print(summary)

Output()

ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


           mean        sd  hdi_3%   hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
mu_1      0.000     0.002  -0.003     0.002       0.00    0.000     567.0   
mu_2      0.000     0.002  -0.001     0.003       0.00    0.001     151.0   
sigma     0.026     0.000   0.025     0.026       0.00    0.000    1620.0   
tau    4228.890  3243.843   7.000  8703.000     389.49   97.414      71.0   

       ess_tail  r_hat  
mu_1      140.0   1.01  
mu_2      137.0   1.02  
sigma    1467.0   1.00  
tau        84.0   1.01  


In [13]:
az.plot_trace(trace)
plt.savefig('/content/drive/MyDrive/plots/change_point_trace.png')
plt.close()

In [14]:
tau_posterior = trace.posterior["tau"].values.flatten()
change_point_idx = int(np.median(tau_posterior))
change_point_date = dates[change_point_idx]
print(f"Estimated change point: {change_point_date}")

Estimated change point: 2003-07-02T00:00:00.000000000


In [15]:
mu_1_posterior = trace.posterior["mu_1"].values.flatten()
mu_2_posterior = trace.posterior["mu_2"].values.flatten()
mean_diff = mu_2_posterior.mean() - mu_1_posterior.mean()
percent_change = (np.exp(mean_diff) - 1) * 100
print(f"Mean log return before change: {mu_1_posterior.mean():.4f}")
print(f"Mean log return after change: {mu_2_posterior.mean():.4f}")
print(f"Percentage change in mean return: {percent_change:.2f}%")

Mean log return before change: 0.0002
Mean log return after change: 0.0002
Percentage change in mean return: -0.00%


In [19]:
events = pd.DataFrame({
    'Date': [
        '2014-11-27',
        '2016-11-30',
        '2018-12-07',
        '2020-03-06',
        '2020-04-12',
        '2022-02-24',
    ],
    'Event': [
        'OPEC refuses to cut production',
        'OPEC agrees to production cut',
        'OPEC+ production cut agreement',
        'OPEC+ talks collapse, price war',
        'OPEC+ historic production cut',
        'Russia-Ukraine conflict begins'
    ]
})
events['Date'] = pd.to_datetime(events['Date'])
events.to_csv('/content/drive/MyDrive/processed/events_data.csv', index=False)


print("\nEvents near the estimated change point:")
for _, row in events.iterrows():
    event_date = row['Date']
    if abs((event_date - change_point_date).days) < 30:
        print(f"Event: {row['Event']} on {event_date.date()}")


Events near the estimated change point:


In [20]:
before_window = brent_df[brent_df['Date'] < change_point_date][-30:]
after_window = brent_df[brent_df['Date'] >= change_point_date][:30]
price_before = before_window['Price'].mean()
price_after = after_window['Price'].mean()
price_change = ((price_after - price_before) / price_before) * 100
print(f"Average price before change: ${price_before:.2f}")
print(f"Average price after change: ${price_after:.2f}")
print(f"Price change: {price_change:.2f}%")

Average price before change: $27.46
Average price after change: $28.81
Price change: 4.93%
